# 🧠 Notebook 2: Causal LSTM Phenology Regressor — Training & Sequential Inference### Project: Days to Harvest Predictor (*Seva Bharat*)This notebook implements the primary SOTA deep sequence model for predicting remaining days to harvest.

In [ ]:
import sysimport torchimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom pathlib import PathPROJECT_ROOT = Path("..").resolve()sys.path.insert(0, str(PROJECT_ROOT))from src.dataset import create_dataloadersfrom src.models.lstm_model import LSTMRegressorfrom src.models.losses import MaskedMSELossfrom src.training.train_lstm import train_lstmfrom src.inference.predict import HarvestPredictorfrom config.config import WEIGHTS_DIRdevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")print(f"PyTorch Version: {torch.__version__}, Device: {device}")

## 1. Load DataLoaders with Causal Temporal Padding & Masking

In [ ]:
train_loader, val_loader, test_loader, meta = create_dataloaders(batch_size=128)print("Metadata:")for k, v in meta.items():    if k not in ['num_cols', 'cat_cols']:        print(f"  {k}: {v}")

## 2. Instantiate Primary Causal LSTM Regressor

In [ ]:
model = LSTMRegressor(    num_dim=meta["num_dim"],    cat_dims=meta["cat_dims"],    hidden_dim=64,    num_layers=2).to(device)print(model)total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)print(f"Total Trainable Parameters: {total_params:,}")

## 3. Load Trained Weights & Evaluate

In [ ]:
weights_path = WEIGHTS_DIR / "lstm_harvest.pth"if weights_path.exists():    model.load_state_dict(torch.load(weights_path, map_location=device))    print("Model weights loaded successfully!")else:    print("Training fresh model...")    model, history = train_lstm(epochs=50, batch_size=128)

## 4. Single-Farm Sequential Inference Walkthrough

In [ ]:
predictor = HarvestPredictor(model_path=weights_path)test_df = pd.read_csv(PROJECT_ROOT / "data/processed/training_data.csv")sample_farm = test_df[test_df["split"] == "test"].groupby(["point_id", "year"]).get_group(list(test_df[test_df["split"] == "test"].groupby(["point_id", "year"]).groups.keys())[0])result = predictor.predict_sequence(sample_farm)print("Inference Result:")for k, v in result.items():    if k != 'trajectory_predictions':        print(f"  {k:35s}: {v}")

## 5. Visualizing the Countdown Trajectory

In [ ]:
plt.figure(figsize=(10, 4.5), dpi=150)steps = np.arange(len(result["trajectory_predictions"]))plt.plot(steps * 5, result["trajectory_predictions"], marker="o", color="crimson", label="Predicted Days Remaining")plt.axhline(0, color="black", linestyle="--")plt.title("Sequential Harvest Countdown Trajectory for Sample Farm", fontsize=12, fontweight="bold")plt.xlabel("Days into Season")plt.ylabel("Estimated Days to Harvest")plt.legend()plt.grid(True, linestyle=":", alpha=0.6)plt.show()